# ETF 市场流动性研究

这个 notebook 拆成了两个层次：

1. **源数据层**：只在第一次运行或显式刷新时，从 Tushare 拉取 ETF 基础信息、最新行情、最新份额、近 20 个交易日行情，并保存为本地 CSV。
2. **分析层**：后续默认直接读取这些 CSV，生成流动性快照、结构统计和结论解读，不再重复访问接口。

这样做的目的是把**数据抓取**和**研究分析**解耦：抓取成本高、受接口限制；分析则应该可以反复重跑、反复调口径。

本 notebook 关注的核心问题：
- 当前场内 ETF 的流动性覆盖率如何
- 流动性是否集中在少数头部产品
- 哪些资产类别更活跃
- 哪些 ETF 兼具高成交额与较高规模换手


## 1. 导入依赖与显示设置

这一部分只负责准备环境：
- `pandas` 用于表格处理
- `tushare` 用于第一次抓数
- `Markdown` / `display` 用于把结果分析直接渲染在 notebook 里

另外统一设置 DataFrame 展示宽度，避免后续结果表被截断。

In [1]:
from pathlib import Path

import pandas as pd
import tushare as ts
from IPython.display import Markdown, display

try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)


## 2. 读取配置并确定目录

这一段做三件事：

1. 从 `research/config.toml` 读取 Tushare token
2. 根据配置中的 `storage.dump_dir` 计算原始 CSV 缓存目录
3. 定义 `REFRESH_SOURCE_DATA` 开关

`REFRESH_SOURCE_DATA = False` 时：
- 如果本地已有原始 CSV，就直接读取
- 如果缺文件，才自动补抓一次

只有在你明确想更新到新的市场快照时，才需要把它改成 `True`。

In [2]:
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'research' else Path.cwd().resolve()
config_candidates = [
    project_root / 'research' / 'config.toml',
    project_root / 'config.toml',
    Path.cwd().resolve() / 'config.toml',
]

config_path = next((path for path in config_candidates if path.exists()), None)
if config_path is None:
    raise FileNotFoundError('未找到 config.toml，请确认 research/config.toml 存在。')

with config_path.open('rb') as f:
    cfg = tomllib.load(f)

ts.set_token(cfg['tushare']['token'])
pro = ts.pro_api()

storage_cfg = cfg.get('storage', {})


def resolve_storage_path(path_value, default_value):
    path = Path(path_value or default_value)
    return path if path.is_absolute() else (project_root / path).resolve()


raw_data_dir = resolve_storage_path(storage_cfg.get('dump_dir'), 'data/raw')
output_dir = (project_root / 'research' / 'outputs').resolve()
raw_data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

REFRESH_SOURCE_DATA = False

SOURCE_FILES = {
    'etf_universe': raw_data_dir / 'etf_universe.csv',
    'latest_daily': raw_data_dir / 'etf_latest_daily.csv',
    'latest_share': raw_data_dir / 'etf_latest_share.csv',
    'recent_daily': raw_data_dir / 'etf_recent_daily.csv',
    'source_context': raw_data_dir / 'etf_source_context.csv',
}

print(f'配置文件: {config_path}')
print(f'原始数据目录: {raw_data_dir}')
print(f'分析结果目录: {output_dir}')
print(f'是否强制刷新源数据: {REFRESH_SOURCE_DATA}')


配置文件: C:\Users\chern\Desktop\my_repos\vibetf\research\config.toml
原始数据目录: C:\Users\chern\Desktop\my_repos\vibetf\data\raw
分析结果目录: C:\Users\chern\Desktop\my_repos\vibetf\research\outputs
是否强制刷新源数据: False


## 3. 源数据管理：优先读 CSV，必要时再抓取

这是这次改动的核心。

### 会缓存哪些文件
- `etf_universe.csv`：场内 ETF 名单
- `etf_latest_daily.csv`：最新可用交易日的 ETF 日线行情
- `etf_latest_share.csv`：最新可用交易日的 ETF 份额数据
- `etf_recent_daily.csv`：最近 20 个交易日的 ETF 日线行情
- `etf_source_context.csv`：这批源数据对应的日期和窗口说明

### 运行逻辑
- 本地 CSV 完整存在时：直接读取
- 缺任何一个文件时：自动抓取并落盘
- 若把 `REFRESH_SOURCE_DATA` 改成 `True`：无论本地是否存在，都会重新抓取

这样后面的研究代码只围绕 CSV 工作，数据抓取只发生在必要的时候。

In [3]:
def paged_fetch(fetch_fn, *, limit=1000, **kwargs):
    frames = []
    offset = 0
    while True:
        df = fetch_fn(offset=offset, limit=limit, **kwargs)
        if df is None or df.empty:
            break
        frames.append(df)
        if len(df) < limit:
            break
        offset += limit
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def get_open_dates(start_date: str, end_date: str) -> list[str]:
    trade_cal = pro.trade_cal(exchange='SSE', start_date=start_date, end_date=end_date, is_open='1')
    return sorted(trade_cal['cal_date'].tolist())


def find_latest_fund_daily_date(open_dates: list[str], search_depth: int = 10):
    for trade_date in reversed(open_dates[-search_depth:]):
        daily = pro.fund_daily(trade_date=trade_date)
        if not daily.empty:
            return trade_date, daily
    raise RuntimeError('最近交易日未取到 ETF 日线数据，请检查 Tushare 权限或日期范围。')


def classify_asset_type(row) -> str:
    text = ' '.join(str(row.get(col) or '') for col in ['extname', 'index_name']).upper()
    if any(keyword in text for keyword in ['货币', '现金', '快线', '银华日利', '华宝添益']):
        return '货币'
    if any(keyword in text for keyword in ['债', '国债', '政金', '信用', '短融', '可转债', '同业存单']):
        return '债券'
    if any(keyword in text for keyword in ['恒生', '纳斯达克', '标普', '日经', '德国', '法国', '港股', 'MSCI', 'S&P', 'NASDAQ', 'DAX', 'CAC', '海外', '东南亚', '中韩']):
        return '跨境'
    if any(keyword in text for keyword in ['黄金', '有色', '豆粕', '石油', '能源', '商品', '饲料', '化工']):
        return '商品'
    return '股票'


def classify_stock_subtype(row) -> str:
    if str(row.get('asset_type') or '') != '股票':
        return '非股票'

    text = ' '.join(str(row.get(col) or '') for col in ['extname', 'index_name']).upper()

    style_strategy_keywords = [
        '红利', '低波', '价值', '成长', '质量', '高股息', '股息', '央企', '国企', 'ESG', '基本面', '治理', '龙头', '品牌', '消费50'
    ]
    industry_keywords = [
        '证券', '券商', '银行', '酒', '煤炭', '医药', '医疗', '创新药', '消费', '食品', '家电', '汽车', '芯片', '半导体', '通信', '5G',
        '军工', '人工智能', '机器人', '电网', '光伏', '新能源车', '新能车', '软件', '传媒', '游戏', '农业', '养殖', '畜牧', '钢铁',
        '化工', '建材', '地产', '家居', '电池', '计算机', '互联网', '云计算', '卫星', '航天', '稀土', '有色', '电力', '环保', '机械'
    ]
    broad_index_keywords = [
        '沪深300', '中证A500', 'A500', '中证500', '中证1000', '中证2000', '上证50', '上证180', '上证380', '深证100', '创业板',
        '创业板50', '科创50', '科创100', '科创200', '中证A50', 'MSCI中国A50', '300ETF', '500ETF', '1000ETF', '2000ETF', '50ETF'
    ]

    if any(keyword in text for keyword in style_strategy_keywords):
        return '风格策略ETF'
    if any(keyword in text for keyword in industry_keywords):
        return '行业ETF'
    if any(keyword in text for keyword in broad_index_keywords):
        return '宽基指数ETF'
    return '主题ETF'


def source_files_ready() -> bool:
    return all(path.exists() for path in SOURCE_FILES.values())


def write_source_context(latest_trade_date: str, recent_trade_dates: list[str], source_mode: str):
    context = pd.DataFrame(
        [
            {'item': 'latest_trade_date', 'value': latest_trade_date},
            {'item': 'recent_window_start', 'value': recent_trade_dates[0]},
            {'item': 'recent_window_end', 'value': recent_trade_dates[-1]},
            {'item': 'recent_window_size', 'value': str(len(recent_trade_dates))},
            {'item': 'source_mode', 'value': source_mode},
            {'item': 'saved_at', 'value': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')},
        ]
    )
    context.to_csv(SOURCE_FILES['source_context'], index=False, encoding='utf-8-sig')


def read_source_context() -> dict:
    context_df = pd.read_csv(SOURCE_FILES['source_context'], dtype=str)
    return dict(zip(context_df['item'], context_df['value']))


def fetch_and_cache_source_data():
    today = pd.Timestamp.today().strftime('%Y%m%d')
    open_dates = get_open_dates(start_date='20250101', end_date=today)

    etf_universe = paged_fetch(
        pro.etf_basic,
        list_status='L',
        fields='ts_code,extname,list_date,exchange,mgr_name,index_code,index_name',
    )
    etf_universe = etf_universe[etf_universe['ts_code'].str.endswith(('.SH', '.SZ'))].drop_duplicates('ts_code').copy()

    latest_trade_date, latest_daily = find_latest_fund_daily_date(open_dates)
    recent_trade_dates = [trade_date for trade_date in open_dates if trade_date <= latest_trade_date][-20:]

    recent_daily_frames = []
    for trade_date in recent_trade_dates:
        day_df = pro.fund_daily(trade_date=trade_date)
        if not day_df.empty:
            recent_daily_frames.append(day_df)
    recent_daily = pd.concat(recent_daily_frames, ignore_index=True)

    latest_share = paged_fetch(pro.fund_share, start_date=latest_trade_date, end_date=latest_trade_date)
    latest_share = latest_share[latest_share['ts_code'].isin(etf_universe['ts_code'])].drop_duplicates('ts_code').copy()

    etf_universe.to_csv(SOURCE_FILES['etf_universe'], index=False, encoding='utf-8-sig')
    latest_daily.to_csv(SOURCE_FILES['latest_daily'], index=False, encoding='utf-8-sig')
    latest_share.to_csv(SOURCE_FILES['latest_share'], index=False, encoding='utf-8-sig')
    recent_daily.to_csv(SOURCE_FILES['recent_daily'], index=False, encoding='utf-8-sig')
    write_source_context(latest_trade_date, recent_trade_dates, source_mode='fetched')

    return etf_universe, latest_daily, latest_share, recent_daily, latest_trade_date, recent_trade_dates, 'fetched'


def load_cached_source_data():
    etf_universe = pd.read_csv(SOURCE_FILES['etf_universe'], dtype={'ts_code': str, 'list_date': str, 'exchange': str, 'mgr_name': str, 'index_code': str, 'index_name': str})
    latest_daily = pd.read_csv(SOURCE_FILES['latest_daily'], dtype={'ts_code': str, 'trade_date': str})
    latest_share = pd.read_csv(SOURCE_FILES['latest_share'], dtype={'ts_code': str, 'trade_date': str, 'market': str, 'fund_type': str})
    recent_daily = pd.read_csv(SOURCE_FILES['recent_daily'], dtype={'ts_code': str, 'trade_date': str})
    context = read_source_context()

    latest_trade_date = str(context.get('latest_trade_date') or latest_daily['trade_date'].astype(str).max())
    recent_trade_dates = sorted(recent_daily['trade_date'].astype(str).unique().tolist())

    return etf_universe, latest_daily, latest_share, recent_daily, latest_trade_date, recent_trade_dates, 'cached'


In [4]:
missing_sources = [name for name, path in SOURCE_FILES.items() if not path.exists()]

if REFRESH_SOURCE_DATA or missing_sources:
    if REFRESH_SOURCE_DATA:
        print('已开启强制刷新：重新从 Tushare 抓取源数据并覆盖本地 CSV。')
    else:
        print(f'检测到缺失的源文件 {missing_sources}，将自动抓取一次并写入本地 CSV。')
    etf_universe, latest_daily, latest_share, recent_daily, latest_trade_date, recent_trade_dates, source_mode = fetch_and_cache_source_data()
else:
    print('检测到本地源数据 CSV 完整，直接读取缓存。')
    etf_universe, latest_daily, latest_share, recent_daily, latest_trade_date, recent_trade_dates, source_mode = load_cached_source_data()

source_mode_text = '本地 CSV 缓存' if source_mode == 'cached' else 'Tushare 抓取后落盘'

print(f'源数据模式: {source_mode_text}')
print(f'最新可用交易日: {latest_trade_date}')
print(f'近 20 日窗口: {recent_trade_dates[0]} ~ {recent_trade_dates[-1]}')
print(f'ETF 名单数量: {len(etf_universe)}')


检测到本地源数据 CSV 完整，直接读取缓存。
源数据模式: 本地 CSV 缓存
最新可用交易日: 20260514
近 20 日窗口: 20260414 ~ 20260514
ETF 名单数量: 1518


## 4. 构建流动性快照

这一段把四类源数据拼成一张统一研究表：

- ETF 名单：提供产品名称、管理人、指数信息
- 最新日线：提供最新成交额和价格
- 最新份额：用于估算产品规模
- 近 20 日日线：用于计算更稳定的均值和中位数

### 这里重点构造了几个指标
- `amount_100m`：最新成交额，单位是亿元
- `avg_amount_100m_20d`：近 20 个交易日平均成交额
- `est_size_100m`：按最新收盘价和基金份额估算的规模，单位是亿元
- `turnover_by_size`：成交额 / 估算规模，可当作粗粒度换手近似

注意：`est_size_100m` 不是基金正式披露的资产净值，只是用于横向比较的近似口径。

In [5]:
recent_daily = recent_daily.copy()
recent_daily['amount_100m'] = recent_daily['amount'] / 100_000

recent_20d_stats = (
    recent_daily.groupby('ts_code')
    .agg(
        avg_amount_100m_20d=('amount_100m', 'mean'),
        median_amount_100m_20d=('amount_100m', 'median'),
        max_amount_100m_20d=('amount_100m', 'max'),
        trading_days_20d=('trade_date', 'count'),
    )
    .reset_index()
)

latest_share = latest_share.rename(columns={'trade_date': 'share_trade_date'})

snapshot = (
    etf_universe
    .merge(latest_daily, on='ts_code', how='left')
    .merge(latest_share[['ts_code', 'share_trade_date', 'fd_share']], on='ts_code', how='left')
    .merge(recent_20d_stats, on='ts_code', how='left')
)

snapshot['asset_type'] = snapshot.apply(classify_asset_type, axis=1)
snapshot['stock_subtype'] = snapshot.apply(classify_stock_subtype, axis=1)
snapshot['amount_100m'] = snapshot['amount'] / 100_000
snapshot['fd_share_100m'] = snapshot['fd_share'] / 10_000
snapshot['est_size_100m'] = snapshot['close'] * snapshot['fd_share'] / 10_000
snapshot['turnover_by_size'] = snapshot['amount_100m'] / snapshot['est_size_100m']
snapshot['listed_year'] = snapshot['list_date'].astype(str).str[:4]

liquid_snapshot = snapshot.dropna(subset=['amount_100m']).copy()

display(
    liquid_snapshot.nlargest(15, 'avg_amount_100m_20d')[
        [
            'ts_code', 'extname', 'asset_type', 'stock_subtype', 'mgr_name', 'amount_100m',
            'avg_amount_100m_20d', 'est_size_100m', 'turnover_by_size'
        ]
    ]
)


,ts_code,extname,asset_type,stock_subtype,mgr_name,amount_100m,avg_amount_100m_20d,est_size_100m,turnover_by_size
731,511360.SH,短融ETF海富通,债券,非股票,海富通基金,203.448525,463.859688,939.630453,0.216520
749,511880.SH,银华日利ETF,货币,非股票,银华基金,90.969290,174.366123,78483.601064,0.001159
758,511990.SH,华宝添益ETF,货币,非股票,华宝基金,59.577733,114.171105,71459.675396,0.000834
719,511100.SH,国债ETF华夏,债券,非股票,华夏基金,105.191117,108.244576,107.042574,0.982704
732,511380.SH,可转债ETF博时,债券,非股票,博时基金,111.328934,98.289652,587.368401,0.189539
1158,551550.SH,华夏中证AAA科技创新公司债ETF,债券,非股票,华夏基金,109.216008,94.652796,155.329822,0.703123
1161,551800.SH,国泰中证AAA科技创新公司债ETF,债券,非股票,国泰基金,81.504856,85.014612,118.259931,0.689201
862,513310.SH,中韩半导体ETF华泰柏瑞,跨境,非股票,华泰柏瑞基金,88.338295,83.925275,137.739592,0.641343
1152,551030.SH,科创债ETF鹏华,债券,非股票,鹏华基金,6.111797,64.700347,179.877031,0.033978
1348,563360.SH,A500ETF华泰柏瑞,股票,宽基指数ETF,华泰柏瑞基金,40.380020,59.125583,326.388965,0.123717


## 5. 市场全景统计

这一部分是从“全市场”的角度看 ETF 流动性，而不是只看头部产品。

### 主要看三类问题
1. **覆盖率**：是不是大多数 ETF 都有交易
2. **分层**：不同成交额门槛下，各有多少只 ETF
3. **结构**：成交主要集中在哪些资产类别、哪些管理人

如果你想快速判断“市场是不是高度头部化”，这一段最重要。

In [6]:
market_stats = pd.DataFrame(
    {
        '指标': [
            '场内 ETF 数量',
            '最新有成交 ETF 数量',
            '最新成交覆盖率',
            '最新总成交额(亿元)',
            '单只成交额中位数(亿元)',
            '单只成交额 90 分位(亿元)',
            '单只成交额 95 分位(亿元)',
            '前 10 大 ETF 成交占比',
            '前 50 大 ETF 成交占比',
        ],
        '数值': [
            len(etf_universe),
            len(liquid_snapshot),
            f"{len(liquid_snapshot) / len(etf_universe):.2%}",
            round(liquid_snapshot['amount_100m'].sum(), 2),
            round(liquid_snapshot['amount_100m'].median(), 2),
            round(liquid_snapshot['amount_100m'].quantile(0.9), 2),
            round(liquid_snapshot['amount_100m'].quantile(0.95), 2),
            f"{liquid_snapshot.nlargest(10, 'amount_100m')['amount_100m'].sum() / liquid_snapshot['amount_100m'].sum():.2%}",
            f"{liquid_snapshot.nlargest(50, 'amount_100m')['amount_100m'].sum() / liquid_snapshot['amount_100m'].sum():.2%}",
        ],
    }
)

threshold_summary = pd.DataFrame(
    [
        {'成交额阈值': '>= 1 亿元', 'ETF 数量': int((liquid_snapshot['amount_100m'] >= 1).sum())},
        {'成交额阈值': '>= 5 亿元', 'ETF 数量': int((liquid_snapshot['amount_100m'] >= 5).sum())},
        {'成交额阈值': '>= 10 亿元', 'ETF 数量': int((liquid_snapshot['amount_100m'] >= 10).sum())},
        {'成交额阈值': '>= 50 亿元', 'ETF 数量': int((liquid_snapshot['amount_100m'] >= 50).sum())},
        {'成交额阈值': '>= 100 亿元', 'ETF 数量': int((liquid_snapshot['amount_100m'] >= 100).sum())},
    ]
)

asset_type_summary = (
    liquid_snapshot.groupby('asset_type')
    .agg(
        etf_count=('ts_code', 'count'),
        total_amount_100m=('amount_100m', 'sum'),
        total_avg_amount_100m_20d=('avg_amount_100m_20d', 'sum'),
        median_amount_100m=('amount_100m', 'median'),
        median_avg_amount_100m_20d=('avg_amount_100m_20d', 'median'),
        avg_amount_100m=('amount_100m', 'mean'),
        avg_avg_amount_100m_20d=('avg_amount_100m_20d', 'mean'),
        p90_amount_100m=('amount_100m', lambda s: s.quantile(0.9)),
    )
    .sort_values('total_avg_amount_100m_20d', ascending=False)
)
asset_type_summary['market_turnover_share'] = asset_type_summary['total_amount_100m'] / asset_type_summary['total_amount_100m'].sum()
asset_type_summary['market_avg20d_share'] = asset_type_summary['total_avg_amount_100m_20d'] / asset_type_summary['total_avg_amount_100m_20d'].sum()
asset_type_summary['count_ge_1_100m'] = liquid_snapshot.groupby('asset_type')['amount_100m'].apply(lambda s: int((s >= 1).sum()))
asset_type_summary['count_ge_10_100m'] = liquid_snapshot.groupby('asset_type')['amount_100m'].apply(lambda s: int((s >= 10).sum()))
asset_type_summary['top10_avg20d_share_in_type'] = (
    liquid_snapshot.groupby('asset_type')['avg_amount_100m_20d'].apply(lambda s: s.nlargest(min(10, len(s))).sum() / s.sum())
)

manager_summary = (
    liquid_snapshot.groupby('mgr_name')
    .agg(etf_count=('ts_code', 'count'), total_amount_100m=('amount_100m', 'sum'))
    .sort_values('total_amount_100m', ascending=False)
    .head(15)
)

display(market_stats)
display(threshold_summary)
display(asset_type_summary)
display(manager_summary)


,指标,数值
0,场内 ETF 数量,1518
1,最新有成交 ETF 数量,1513
2,最新成交覆盖率,99.67%
3,最新总成交额(亿元),4868.31
4,单只成交额中位数(亿元),0.23
5,单只成交额 90 分位(亿元),5.0
6,单只成交额 95 分位(亿元),14.51
7,前 10 大 ETF 成交占比,22.58%
8,前 50 大 ETF 成交占比,60.05%


,成交额阈值,ETF 数量
0,>= 1 亿元,399
1,>= 5 亿元,152
2,>= 10 亿元,92
3,>= 50 亿元,25
4,>= 100 亿元,6


,etf_count,total_amount_100m,total_avg_amount_100m_20d,median_amount_100m,median_avg_amount_100m_20d,avg_amount_100m,avg_avg_amount_100m_20d,p90_amount_100m,market_turnover_share,market_avg20d_share,count_ge_1_100m,count_ge_10_100m,top10_avg20d_share_in_type
asset_type,,,,,,,,,,,,,
债券,53,2047.426795,2136.849922,27.223234,27.883076,38.630694,40.317923,99.781391,0.420562,0.447622,48,40,0.533832
股票,1057,1679.607658,1482.770037,0.160156,0.148781,1.589033,1.402810,2.684200,0.345008,0.310607,214,33,0.339706
跨境,247,775.098998,634.958732,0.616887,0.606483,3.138053,2.570683,5.734746,0.159213,0.133009,97,12,0.509841
货币,52,185.944335,325.085602,0.136191,0.109626,3.575853,6.251646,2.409784,0.038195,0.068098,7,3,0.981189
商品,104,180.230511,194.120651,0.398449,0.414401,1.732986,1.866545,4.160449,0.037021,0.040664,33,4,0.586814


,etf_count,total_amount_100m
mgr_name,,
华夏基金,125,768.354637
易方达基金,130,530.882608
华泰柏瑞基金,58,356.144479
南方基金,78,306.263006
国泰基金,77,298.903289
海富通基金,9,291.716331
广发基金,78,255.029042
博时基金,63,175.188076
嘉实基金,61,162.106503


## 6. 近 20 日均成交额头部 ETF

这里把全市场 ETF 按 **近 20 日均成交额** 排序，只取前 10。

这样做的目的是弱化单日冲高或事件驱动的影响，更接近“稳定流动性”的定义。

如果一个 ETF 在这个榜单靠前，通常说明它不是偶发活跃，而是在最近一个交易窗口里持续具备较强成交承载能力。


In [7]:
top_20d_avg = liquid_snapshot.nlargest(10, 'avg_amount_100m_20d')[
    [
        'ts_code', 'extname', 'asset_type', 'stock_subtype', 'mgr_name',
        'amount_100m', 'avg_amount_100m_20d', 'median_amount_100m_20d', 'est_size_100m', 'turnover_by_size'
    ]
].copy()
top_20d_avg = top_20d_avg.rename(columns={
    'amount_100m': '最新成交额(亿元)',
    'avg_amount_100m_20d': '近20日均成交额(亿元)',
    'median_amount_100m_20d': '近20日中位成交额(亿元)',
    'est_size_100m': '估算规模(亿元)',
    'turnover_by_size': '成交额/规模',
})

display(top_20d_avg)


,ts_code,extname,asset_type,stock_subtype,mgr_name,最新成交额(亿元),近20日均成交额(亿元),近20日中位成交额(亿元),估算规模(亿元),成交额/规模
731,511360.SH,短融ETF海富通,债券,非股票,海富通基金,203.448525,463.859688,438.829356,939.630453,0.216520
749,511880.SH,银华日利ETF,货币,非股票,银华基金,90.969290,174.366123,176.005632,78483.601064,0.001159
758,511990.SH,华宝添益ETF,货币,非股票,华宝基金,59.577733,114.171105,102.579439,71459.675396,0.000834
719,511100.SH,国债ETF华夏,债券,非股票,华夏基金,105.191117,108.244576,106.500215,107.042574,0.982704
732,511380.SH,可转债ETF博时,债券,非股票,博时基金,111.328934,98.289652,93.479481,587.368401,0.189539
1158,551550.SH,华夏中证AAA科技创新公司债ETF,债券,非股票,华夏基金,109.216008,94.652796,112.741766,155.329822,0.703123
1161,551800.SH,国泰中证AAA科技创新公司债ETF,债券,非股票,国泰基金,81.504856,85.014612,91.514286,118.259931,0.689201
862,513310.SH,中韩半导体ETF华泰柏瑞,跨境,非股票,华泰柏瑞基金,88.338295,83.925275,67.865007,137.739592,0.641343
1152,551030.SH,科创债ETF鹏华,债券,非股票,鹏华基金,6.111797,64.700347,64.575961,179.877031,0.033978
1348,563360.SH,A500ETF华泰柏瑞,股票,宽基指数ETF,华泰柏瑞基金,40.380020,59.125583,60.032306,326.388965,0.123717


## 7. 按 asset_type 分组统计与组内 Top 10

这一部分把市场再往下拆一层：先按 `asset_type` 分类，再看每一类内部谁最活跃。

这里会输出两类结果：
- **按 asset_type 的分类统计**：每类有多少只 ETF、总成交额、近 20 日均成交额汇总，以及各类内部高流动性产品数量
- **每个 asset_type 内部按近 20 日均成交额排序的 Top 10**：便于直接定位每个类别的核心交易载体

为了让每个市场更容易单独阅读，下面在每个 `asset_type` 的 Top 10 表前面，都会先给一段市场摘要，包含该类 ETF 的数量、总成交额、近 20 日均总成交额、总规模和中位规模等信息。

如果你的研究目标不是“全市场最强 ETF”，而是“股票 / 债券 / 跨境 / 商品 / 货币各自谁最强”，这一段最直接。


In [8]:
asset_type_market_brief = (
    liquid_snapshot.groupby('asset_type')
    .agg(
        etf_count=('ts_code', 'count'),
        total_amount_100m=('amount_100m', 'sum'),
        total_avg_amount_100m_20d=('avg_amount_100m_20d', 'sum'),
        median_avg_amount_100m_20d=('avg_amount_100m_20d', 'median'),
        total_est_size_100m=('est_size_100m', 'sum'),
        median_est_size_100m=('est_size_100m', 'median'),
        median_turnover_by_size=('turnover_by_size', 'median'),
    )
    .sort_values('total_avg_amount_100m_20d', ascending=False)
)
asset_type_market_brief['count_ge_1_100m'] = liquid_snapshot.groupby('asset_type')['amount_100m'].apply(lambda s: int((s >= 1).sum()))
asset_type_market_brief['count_ge_10_100m'] = liquid_snapshot.groupby('asset_type')['amount_100m'].apply(lambda s: int((s >= 10).sum()))

asset_type_top10 = (
    liquid_snapshot.sort_values(['asset_type', 'avg_amount_100m_20d', 'amount_100m'], ascending=[True, False, False])
    .groupby('asset_type', group_keys=False)
    .head(10)
    .copy()
)
asset_type_top10['rank_in_asset_type'] = asset_type_top10.groupby('asset_type')['avg_amount_100m_20d'].rank(method='first', ascending=False).astype(int)
asset_type_top10 = asset_type_top10[
    [
        'asset_type', 'rank_in_asset_type', 'ts_code', 'extname', 'mgr_name',
        'amount_100m', 'avg_amount_100m_20d', 'est_size_100m', 'turnover_by_size'
    ]
]
asset_type_top10 = asset_type_top10.rename(columns={
    'amount_100m': '最新成交额(亿元)',
    'avg_amount_100m_20d': '近20日均成交额(亿元)',
    'est_size_100m': '估算规模(亿元)',
    'turnover_by_size': '成交额/规模',
})

for asset_type, group in asset_type_top10.groupby('asset_type', sort=False):
    brief = asset_type_market_brief.loc[asset_type]
    summary_md = f"""
### {asset_type} ETF 市场概览

- 标的数量：**{int(brief['etf_count'])}** 只
- 最新交易日总成交额：**{brief['total_amount_100m']:.2f}** 亿元
- 近20日均成交额合计：**{brief['total_avg_amount_100m_20d']:.2f}** 亿元
- 近20日均成交额中位数：**{brief['median_avg_amount_100m_20d']:.2f}** 亿元
- 估算总规模：**{brief['total_est_size_100m']:.2f}** 亿元
- 估算规模中位数：**{brief['median_est_size_100m']:.2f}** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**{int(brief['count_ge_1_100m'])}** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**{int(brief['count_ge_10_100m'])}** 只
- 成交额/规模 中位数：**{brief['median_turnover_by_size']:.4f}**

### {asset_type} ETF 近20日均成交额 Top 10
"""
    display(Markdown(summary_md))
    display(group.reset_index(drop=True))



### 债券 ETF 市场概览

- 标的数量：**53** 只
- 最新交易日总成交额：**2047.43** 亿元
- 近20日均成交额合计：**2136.85** 亿元
- 近20日均成交额中位数：**27.88** 亿元
- 估算总规模：**7945.79** 亿元
- 估算规模中位数：**105.33** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**48** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**40** 只
- 成交额/规模 中位数：**0.2817**

### 债券 ETF 近20日均成交额 Top 10


,asset_type,rank_in_asset_type,ts_code,extname,mgr_name,最新成交额(亿元),近20日均成交额(亿元),估算规模(亿元),成交额/规模
0,债券,1,511360.SH,短融ETF海富通,海富通基金,203.448525,463.859688,939.630453,0.216520
1,债券,2,511100.SH,国债ETF华夏,华夏基金,105.191117,108.244576,107.042574,0.982704
2,债券,3,511380.SH,可转债ETF博时,博时基金,111.328934,98.289652,587.368401,0.189539
3,债券,4,551550.SH,华夏中证AAA科技创新公司债ETF,华夏基金,109.216008,94.652796,155.329822,0.703123
4,债券,5,551800.SH,国泰中证AAA科技创新公司债ETF,国泰基金,81.504856,85.014612,118.259931,0.689201
5,债券,6,551030.SH,科创债ETF鹏华,鹏华基金,6.111797,64.700347,179.877031,0.033978
6,债券,7,511070.SH,公司债ETF南方,南方基金,54.232438,58.098154,193.941040,0.279634
7,债券,8,551060.SH,科创债ETF中银,中银基金,123.953327,56.133266,72.087188,1.719492
8,债券,9,159700.SZ,科创债ETF南方,南方基金,44.098417,56.101044,114.617482,0.384744
9,债券,10,511110.SH,公司债ETF易方达,易方达基金,104.350525,55.625247,241.772963,0.431605



### 商品 ETF 市场概览

- 标的数量：**104** 只
- 最新交易日总成交额：**180.23** 亿元
- 近20日均成交额合计：**194.12** 亿元
- 近20日均成交额中位数：**0.41** 亿元
- 估算总规模：**5257.72** 亿元
- 估算规模中位数：**8.95** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**33** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**4** 只
- 成交额/规模 中位数：**0.0546**

### 商品 ETF 近20日均成交额 Top 10


,asset_type,rank_in_asset_type,ts_code,extname,mgr_name,最新成交额(亿元),近20日均成交额(亿元),估算规模(亿元),成交额/规模
0,商品,1,518880.SH,黄金ETF华安,华安基金,29.034989,37.543666,1140.102816,0.025467
1,商品,2,512400.SH,有色金属ETF南方,南方基金,13.452762,11.604672,280.457931,0.047967
2,商品,3,159755.SZ,电池ETF广发,广发基金,10.305499,11.151216,158.004513,0.065223
3,商品,4,159981.SZ,能源化工ETF建信,建信基金,4.114175,10.615778,37.494816,0.109727
4,商品,5,159870.SZ,化工ETF鹏华,鹏华基金,13.925178,10.064301,255.033919,0.054601
5,商品,6,159934.SZ,黄金ETF易方达,易方达基金,7.228693,9.256046,424.125947,0.017044
6,商品,7,159937.SZ,黄金ETF博时,博时基金,7.295593,7.260501,490.242530,0.014882
7,商品,8,159566.SZ,储能电池ETF易方达,易方达基金,6.673157,6.721024,75.478955,0.088411
8,商品,9,518800.SH,黄金ETF国泰,国泰基金,3.445445,4.928697,407.471784,0.008456
9,商品,10,159980.SZ,有色ETF大成,大成基金,5.615127,4.766722,72.295402,0.077669



### 股票 ETF 市场概览

- 标的数量：**1057** 只
- 最新交易日总成交额：**1679.61** 亿元
- 近20日均成交额合计：**1482.77** 亿元
- 近20日均成交额中位数：**0.15** 亿元
- 估算总规模：**26454.29** 亿元
- 估算规模中位数：**2.81** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**214** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**33** 只
- 成交额/规模 中位数：**0.0581**

### 股票 ETF 近20日均成交额 Top 10


,asset_type,rank_in_asset_type,ts_code,extname,mgr_name,最新成交额(亿元),近20日均成交额(亿元),估算规模(亿元),成交额/规模
0,股票,1,563360.SH,A500ETF华泰柏瑞,华泰柏瑞基金,40.380020,59.125583,326.388965,0.123717
1,股票,2,510500.SH,中证500ETF南方,南方基金,79.003993,58.809961,441.316566,0.179019
2,股票,3,159352.SZ,A500ETF南方,南方基金,46.691911,53.796107,317.062447,0.147264
3,股票,4,510300.SH,沪深300ETF华泰柏瑞,华泰柏瑞基金,51.186635,53.787211,1585.655230,0.032281
4,股票,5,513090.SH,香港证券ETF易方达,易方达基金,48.909250,53.099038,217.138070,0.225245
5,股票,6,159338.SZ,中证A500ETF国泰,国泰基金,44.099714,52.013329,248.868266,0.177201
6,股票,7,159915.SZ,创业板ETF易方达,易方达基金,59.065714,49.516870,477.397740,0.123724
7,股票,8,588000.SH,科创50ETF华夏,华夏基金,64.122166,48.170049,686.771828,0.093367
8,股票,9,588200.SH,科创芯片ETF嘉实,嘉实基金,56.429350,37.926578,473.684336,0.119129
9,股票,10,512050.SH,A500ETF华夏,华夏基金,27.306991,37.461365,213.510948,0.127895



### 货币 ETF 市场概览

- 标的数量：**52** 只
- 最新交易日总成交额：**185.94** 亿元
- 近20日均成交额合计：**325.09** 亿元
- 近20日均成交额中位数：**0.11** 亿元
- 估算总规模：**162715.34** 亿元
- 估算规模中位数：**21.36** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**7** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**3** 只
- 成交额/规模 中位数：**0.0122**

### 货币 ETF 近20日均成交额 Top 10


,asset_type,rank_in_asset_type,ts_code,extname,mgr_name,最新成交额(亿元),近20日均成交额(亿元),估算规模(亿元),成交额/规模
0,货币,1,511880.SH,银华日利ETF,银华基金,90.969290,174.366123,78483.601064,0.001159
1,货币,2,511990.SH,华宝添益ETF,华宝基金,59.577733,114.171105,71459.675396,0.000834
2,货币,3,511660.SH,货币ETF建信,建信基金,11.815407,11.784766,8487.238925,0.001392
3,货币,4,159001.SZ,货币ETF易方达,易方达基金,3.785404,6.709123,NaN,NaN
4,货币,5,159201.SZ,自由现金流ETF华夏,华夏基金,5.956079,5.020851,204.862296,0.029074
5,货币,6,159232.SZ,自由现金流ETF南方,南方基金,2.069831,2.265282,45.295910,0.045696
6,货币,7,159399.SZ,现金流ETF国泰,国泰基金,2.447556,1.833501,53.296303,0.045924
7,货币,8,159235.SZ,中证现金流ETF大成,大成基金,0.774137,0.976124,39.783288,0.019459
8,货币,9,159222.SZ,自由现金流ETF易方达,易方达基金,0.980775,0.955317,21.852535,0.044882
9,货币,10,563760.SH,全指现金流ETF中银,中银基金,0.723847,0.888288,12.344371,0.058638



### 跨境 ETF 市场概览

- 标的数量：**247** 只
- 最新交易日总成交额：**775.10** 亿元
- 近20日均成交额合计：**634.96** 亿元
- 近20日均成交额中位数：**0.61** 亿元
- 估算总规模：**9712.14** 亿元
- 估算规模中位数：**6.32** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**97** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**12** 只
- 成交额/规模 中位数：**0.0707**

### 跨境 ETF 近20日均成交额 Top 10


,asset_type,rank_in_asset_type,ts_code,extname,mgr_name,最新成交额(亿元),近20日均成交额(亿元),估算规模(亿元),成交额/规模
0,跨境,1,513310.SH,中韩半导体ETF华泰柏瑞,华泰柏瑞基金,88.338295,83.925275,137.739592,0.641343
1,跨境,2,513120.SH,港股创新药ETF广发,广发基金,35.365864,44.104985,249.377810,0.141816
2,跨境,3,513180.SH,恒生科技ETF华夏,华夏基金,64.770200,40.470918,538.038926,0.120382
3,跨境,4,513130.SH,恒生科技ETF华泰柏瑞,华泰柏瑞基金,63.352773,40.149974,486.415410,0.130244
4,跨境,5,159570.SZ,港股通创新药ETF汇添富,汇添富基金,17.211777,23.550276,286.958726,0.059980
5,跨境,6,513330.SH,恒生互联网ETF华夏,华夏基金,41.915623,21.469216,342.440853,0.122403
6,跨境,7,513050.SH,中概互联网ETF易方达,易方达基金,40.315510,21.187658,411.600625,0.097948
7,跨境,8,159792.SZ,港股通互联网ETF富国,富国基金,33.591853,17.333247,599.505552,0.056033
8,跨境,9,159518.SZ,标普油气ETF嘉实,嘉实基金,7.611117,15.871011,23.066893,0.329958
9,跨境,10,159740.SZ,恒生科技ETF大成,大成基金,22.892778,15.665143,200.864030,0.113972


## 8. 股票 ETF 细分类别分析

对股票 ETF 再往下细分时，这里采用一个便于研究的启发式分类：
- **宽基指数ETF**：如沪深300、中证500、中证1000、A500、上证50、科创50、创业板等
- **行业ETF**：如证券、银行、芯片、半导体、通信、医药、消费、军工等
- **风格策略ETF**：如红利、低波、价值、成长、央企、国企等
- **主题ETF**：不属于前三类的股票主题产品

下面会先看股票 ETF 各子类别的统计，再看每个子类别按近 20 日均成交额排序的 Top 10。

和前面的 `asset_type` 一样，这里对每个股票子类别也会先输出一段市场概览，包含标的数量、总成交额、近 20 日均成交额合计、总规模和中位规模等指标，再展示对应的 Top 10。


In [9]:
stock_snapshot = liquid_snapshot[liquid_snapshot['asset_type'] == '股票'].copy()

stock_subtype_summary = (
    stock_snapshot.groupby('stock_subtype')
    .agg(
        etf_count=('ts_code', 'count'),
        total_amount_100m=('amount_100m', 'sum'),
        total_avg_amount_100m_20d=('avg_amount_100m_20d', 'sum'),
        median_avg_amount_100m_20d=('avg_amount_100m_20d', 'median'),
        avg_avg_amount_100m_20d=('avg_amount_100m_20d', 'mean'),
        total_est_size_100m=('est_size_100m', 'sum'),
        median_est_size_100m=('est_size_100m', 'median'),
        median_turnover_by_size=('turnover_by_size', 'median'),
    )
    .sort_values('total_avg_amount_100m_20d', ascending=False)
)
stock_subtype_summary['avg20d_share_within_stock'] = stock_subtype_summary['total_avg_amount_100m_20d'] / stock_subtype_summary['total_avg_amount_100m_20d'].sum()
stock_subtype_summary['top10_avg20d_share_in_subtype'] = (
    stock_snapshot.groupby('stock_subtype')['avg_amount_100m_20d'].apply(lambda s: s.nlargest(min(10, len(s))).sum() / s.sum())
)
stock_subtype_summary['count_ge_1_100m'] = stock_snapshot.groupby('stock_subtype')['amount_100m'].apply(lambda s: int((s >= 1).sum()))
stock_subtype_summary['count_ge_10_100m'] = stock_snapshot.groupby('stock_subtype')['amount_100m'].apply(lambda s: int((s >= 10).sum()))

display(stock_subtype_summary)

stock_subtype_top10 = (
    stock_snapshot.sort_values(['stock_subtype', 'avg_amount_100m_20d', 'amount_100m'], ascending=[True, False, False])
    .groupby('stock_subtype', group_keys=False)
    .head(10)
    .copy()
)
stock_subtype_top10['rank_in_stock_subtype'] = stock_subtype_top10.groupby('stock_subtype')['avg_amount_100m_20d'].rank(method='first', ascending=False).astype(int)
stock_subtype_top10 = stock_subtype_top10[
    [
        'stock_subtype', 'rank_in_stock_subtype', 'ts_code', 'extname', 'index_name', 'mgr_name',
        'amount_100m', 'avg_amount_100m_20d', 'est_size_100m', 'turnover_by_size'
    ]
]
stock_subtype_top10 = stock_subtype_top10.rename(columns={
    'amount_100m': '最新成交额(亿元)',
    'avg_amount_100m_20d': '近20日均成交额(亿元)',
    'est_size_100m': '估算规模(亿元)',
    'turnover_by_size': '成交额/规模',
})

for stock_subtype, group in stock_subtype_top10.groupby('stock_subtype', sort=False):
    brief = stock_subtype_summary.loc[stock_subtype]
    summary_md = f"""
### {stock_subtype} 市场概览

- 标的数量：**{int(brief['etf_count'])}** 只
- 最新交易日总成交额：**{brief['total_amount_100m']:.2f}** 亿元
- 近20日均成交额合计：**{brief['total_avg_amount_100m_20d']:.2f}** 亿元
- 近20日均成交额中位数：**{brief['median_avg_amount_100m_20d']:.2f}** 亿元
- 估算总规模：**{brief['total_est_size_100m']:.2f}** 亿元
- 估算规模中位数：**{brief['median_est_size_100m']:.2f}** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**{int(brief['count_ge_1_100m'])}** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**{int(brief['count_ge_10_100m'])}** 只
- 成交额/规模 中位数：**{brief['median_turnover_by_size']:.4f}**

### {stock_subtype} 近20日均成交额 Top 10
"""
    display(Markdown(summary_md))
    display(group.reset_index(drop=True))


,etf_count,total_amount_100m,total_avg_amount_100m_20d,median_avg_amount_100m_20d,avg_avg_amount_100m_20d,total_est_size_100m,median_est_size_100m,median_turnover_by_size,avg20d_share_within_stock,top10_avg20d_share_in_subtype,count_ge_1_100m,count_ge_10_100m
stock_subtype,,,,,,,,,,,,
宽基指数ETF,305,785.087265,765.584341,0.115183,2.510113,11689.401809,2.241393,0.056199,0.516320,0.623256,53,17
行业ETF,377,744.089965,586.621030,0.353194,1.556024,10093.641769,5.406025,0.073829,0.395625,0.389530,121,16
主题ETF,234,86.131145,72.113816,0.091003,0.308179,2604.528091,1.791824,0.057026,0.048635,0.366171,22,0
风格策略ETF,141,64.299283,58.450850,0.065438,0.414545,2066.720444,1.572768,0.036825,0.039420,0.630866,18,0



### 主题ETF 市场概览

- 标的数量：**234** 只
- 最新交易日总成交额：**86.13** 亿元
- 近20日均成交额合计：**72.11** 亿元
- 近20日均成交额中位数：**0.09** 亿元
- 估算总规模：**2604.53** 亿元
- 估算规模中位数：**1.79** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**22** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**0** 只
- 成交额/规模 中位数：**0.0570**

### 主题ETF 近20日均成交额 Top 10


,stock_subtype,rank_in_stock_subtype,ts_code,extname,index_name,mgr_name,最新成交额(亿元),近20日均成交额(亿元),估算规模(亿元),成交额/规模
0,主题ETF,1,562800.SH,稀有金属ETF嘉实,中证稀有金属主题指数,嘉实基金,5.730474,4.803562,75.584715,0.075815
1,主题ETF,2,510210.SH,上证指数ETF富国,上证综合指数,富国基金,5.940847,4.139136,62.179858,0.095543
2,主题ETF,3,159766.SZ,旅游ETF富国,中证旅游主题指数,富国基金,2.211942,3.296693,39.794904,0.055584
3,主题ETF,4,159781.SZ,科创创业ETF易方达,中证科创创业50指数,易方达基金,4.434812,2.835292,132.554805,0.033456
4,主题ETF,5,159851.SZ,金融科技ETF华宝,中证金融科技主题指数,华宝基金,3.535322,2.783055,76.313500,0.046326
5,主题ETF,6,159667.SZ,工业母机ETF国泰,中证机床指数,国泰基金,2.155366,2.134366,21.292984,0.101224
6,主题ETF,7,159608.SZ,稀有金属ETF广发,中证稀有金属主题指数,广发基金,2.930154,2.072013,54.090051,0.054172
7,主题ETF,8,589680.SH,科创综指ETF鹏华,上证科创板综合指数,鹏华基金,1.989421,1.615758,10.938324,0.181876
8,主题ETF,9,589800.SH,科创综指ETF易方达,上证科创板综合指数,易方达基金,1.714025,1.371885,16.454002,0.104171
9,主题ETF,10,513400.SH,道琼斯ETF鹏华,道琼斯工业平均指数,鹏华基金,0.924390,1.354221,26.207466,0.035272



### 宽基指数ETF 市场概览

- 标的数量：**305** 只
- 最新交易日总成交额：**785.09** 亿元
- 近20日均成交额合计：**765.58** 亿元
- 近20日均成交额中位数：**0.12** 亿元
- 估算总规模：**11689.40** 亿元
- 估算规模中位数：**2.24** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**53** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**17** 只
- 成交额/规模 中位数：**0.0562**

### 宽基指数ETF 近20日均成交额 Top 10


,stock_subtype,rank_in_stock_subtype,ts_code,extname,index_name,mgr_name,最新成交额(亿元),近20日均成交额(亿元),估算规模(亿元),成交额/规模
0,宽基指数ETF,1,563360.SH,A500ETF华泰柏瑞,中证A500指数,华泰柏瑞基金,40.380020,59.125583,326.388965,0.123717
1,宽基指数ETF,2,510500.SH,中证500ETF南方,中证小盘500指数,南方基金,79.003993,58.809961,441.316566,0.179019
2,宽基指数ETF,3,159352.SZ,A500ETF南方,中证A500指数,南方基金,46.691911,53.796107,317.062447,0.147264
3,宽基指数ETF,4,510300.SH,沪深300ETF华泰柏瑞,沪深300指数,华泰柏瑞基金,51.186635,53.787211,1585.655230,0.032281
4,宽基指数ETF,5,159338.SZ,中证A500ETF国泰,中证A500指数,国泰基金,44.099714,52.013329,248.868266,0.177201
5,宽基指数ETF,6,159915.SZ,创业板ETF易方达,创业板指数,易方达基金,59.065714,49.516870,477.397740,0.123724
6,宽基指数ETF,7,588000.SH,科创50ETF华夏,上证科创板50成份指数,华夏基金,64.122166,48.170049,686.771828,0.093367
7,宽基指数ETF,8,512050.SH,A500ETF华夏,中证A500指数,华夏基金,27.306991,37.461365,213.510948,0.127895
8,宽基指数ETF,9,510050.SH,上证50ETF华夏,上证50指数,华夏基金,32.308503,34.742465,360.176485,0.089702
9,宽基指数ETF,10,512100.SH,中证1000ETF南方,中证1000指数,南方基金,22.382074,29.731753,117.276349,0.190849



### 行业ETF 市场概览

- 标的数量：**377** 只
- 最新交易日总成交额：**744.09** 亿元
- 近20日均成交额合计：**586.62** 亿元
- 近20日均成交额中位数：**0.35** 亿元
- 估算总规模：**10093.64** 亿元
- 估算规模中位数：**5.41** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**121** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**16** 只
- 成交额/规模 中位数：**0.0738**

### 行业ETF 近20日均成交额 Top 10


,stock_subtype,rank_in_stock_subtype,ts_code,extname,index_name,mgr_name,最新成交额(亿元),近20日均成交额(亿元),估算规模(亿元),成交额/规模
0,行业ETF,1,513090.SH,香港证券ETF易方达,中证香港证券投资主题指数,易方达基金,48.909250,53.099038,217.138070,0.225245
1,行业ETF,2,588200.SH,科创芯片ETF嘉实,上证科创板芯片指数,嘉实基金,56.429350,37.926578,473.684336,0.119129
2,行业ETF,3,515880.SH,通信ETF国泰,中证全指通信设备指数,国泰基金,37.473176,26.782478,291.536335,0.128537
3,行业ETF,4,159206.SZ,卫星ETF永赢,国证商用卫星通信产业指数,永赢基金,27.686740,18.733523,250.424119,0.110559
4,行业ETF,5,512880.SH,证券ETF国泰,中证全指证券公司指数,国泰基金,21.738905,18.241167,560.050442,0.038816
5,行业ETF,6,159516.SZ,半导体设备ETF国泰,中证半导体材料设备主题指数,国泰基金,23.139089,17.597772,225.581315,0.102575
6,行业ETF,7,159326.SZ,电网设备ETF华夏,中证电网设备主题指数,华夏基金,21.754113,17.559184,327.147275,0.066496
7,行业ETF,8,512480.SH,半导体ETF国联安,中证全指半导体产品与设备指数,国联安基金,18.246841,16.142246,198.845264,0.091764
8,行业ETF,9,159363.SZ,创业板人工智能ETF华宝,创业板人工智能指数,华宝基金,15.398703,11.628819,73.547644,0.209370
9,行业ETF,10,159995.SZ,芯片ETF华夏,国证半导体芯片指数,华夏基金,11.655007,10.795945,262.838465,0.044343



### 风格策略ETF 市场概览

- 标的数量：**141** 只
- 最新交易日总成交额：**64.30** 亿元
- 近20日均成交额合计：**58.45** 亿元
- 近20日均成交额中位数：**0.07** 亿元
- 估算总规模：**2066.72** 亿元
- 估算规模中位数：**1.57** 亿元
- 单日成交额 >= 1 亿元的 ETF 数量：**18** 只
- 单日成交额 >= 10 亿元的 ETF 数量：**0** 只
- 成交额/规模 中位数：**0.0368**

### 风格策略ETF 近20日均成交额 Top 10


,stock_subtype,rank_in_stock_subtype,ts_code,extname,index_name,mgr_name,最新成交额(亿元),近20日均成交额(亿元),估算规模(亿元),成交额/规模
0,风格策略ETF,1,512890.SH,红利低波ETF华泰柏瑞,中证红利低波动指数,华泰柏瑞基金,5.600072,6.742943,310.618183,0.018029
1,风格策略ETF,2,159967.SZ,创业板成长ETF华夏,创业板动量成长指数,华夏基金,7.138376,6.145811,48.436550,0.147376
2,风格策略ETF,3,512710.SH,军工龙头ETF富国,中证军工龙头指数,富国基金,6.482331,4.604653,91.066196,0.071183
3,风格策略ETF,4,510880.SH,红利ETF华泰柏瑞,上证红利指数,华泰柏瑞基金,3.305069,4.480735,195.271118,0.016926
4,风格策略ETF,5,515180.SH,红利ETF易方达,中证红利指数,易方达基金,2.575793,3.128004,143.664324,0.017929
5,风格策略ETF,6,515080.SH,中证红利ETF招商,中证红利指数,招商基金,3.031671,3.001989,98.185938,0.030877
6,风格策略ETF,7,159259.SZ,成长ETF易方达,国证成长100指数,易方达基金,5.263894,2.958692,26.494067,0.198682
7,风格策略ETF,8,563020.SH,红利低波ETF易方达,中证红利低波动指数,易方达基金,2.240771,2.341888,96.250437,0.023281
8,风格策略ETF,9,159263.SZ,价值ETF易方达,国证价值100指数,易方达基金,2.055592,1.919857,42.517759,0.048347
9,风格策略ETF,10,588020.SH,科创成长ETF易方达,上证科创板成长指数,易方达基金,1.973613,1.550096,11.291335,0.174790


## 9. 结果分析

这一段不只是罗列数字，而是把数字翻译成研究结论。

阅读时可以重点关注：
- **覆盖率高不高**：说明 ETF 市场是否普遍可交易
- **中位数和总量差距大不大**：说明长尾是否明显
- **前 10 / 前 50 占比高不高**：说明头部集中度
- **债券 / 股票 / 跨境 / 商品谁更强**：说明当前资金偏好的方向
- **近20日均成交额头部 ETF 是否稳定**：说明流动性是事件性还是常态性
- **股票 ETF 的宽基 / 行业 / 风格 / 主题谁更强**：说明股票内部结构


In [10]:
top_asset_type = asset_type_summary.index[0]
top_asset_turnover_share = asset_type_summary.iloc[0]['market_turnover_share']
top_20d_name = top_20d_avg.iloc[0]['extname']
top_20d_turnover = top_20d_avg.iloc[0]['近20日均成交额(亿元)']
small_size_share = (liquid_snapshot['amount_100m'] < 1).mean()
asset_type_leaders = asset_type_top10.sort_values(['asset_type', 'rank_in_asset_type']).groupby('asset_type').first()
asset_type_leaders_text = '；'.join(
    f"{asset_type}由 {row['extname']} 领跑（近20日均成交额 {row['近20日均成交额(亿元)']:.2f} 亿元）"
    for asset_type, row in asset_type_leaders.iterrows()
)
stock_subtype_leaders = stock_subtype_top10.sort_values(['stock_subtype', 'rank_in_stock_subtype']).groupby('stock_subtype').first()
stock_subtype_leaders_text = '；'.join(
    f"{stock_subtype}由 {row['extname']} 领跑（近20日均成交额 {row['近20日均成交额(亿元)']:.2f} 亿元）"
    for stock_subtype, row in stock_subtype_leaders.iterrows()
)

analysis_md = f"""
### 核心结论

1. **本次分析基于 {latest_trade_date} 的快照，源数据来自 {source_mode_text}。**
2. **场内 ETF 共 {len(etf_universe)} 只，其中 {len(liquid_snapshot)} 只有成交，覆盖率 {len(liquid_snapshot) / len(etf_universe):.2%}。** 这说明 ETF 市场整体上是“广覆盖”的，大多数产品在最新交易日都有成交记录。
3. **最新交易日 ETF 总成交额约 {liquid_snapshot['amount_100m'].sum():.2f} 亿元，但单只 ETF 成交额中位数只有 {liquid_snapshot['amount_100m'].median():.2f} 亿元。** 总量很大，但典型单只产品的流动性并不高，市场呈现明显长尾。
4. **前 10 大 ETF 占全市场成交额 {liquid_snapshot.nlargest(10, 'amount_100m')['amount_100m'].sum() / liquid_snapshot['amount_100m'].sum():.2%}，前 50 大占 {liquid_snapshot.nlargest(50, 'amount_100m')['amount_100m'].sum() / liquid_snapshot['amount_100m'].sum():.2%}。** 这意味着流动性高度集中在头部产品，研究和交易时不能只看“是否上市”，还要看是否进入核心流动性池。
5. **有 {small_size_share:.2%} 的有成交 ETF 当日成交额低于 1 亿元。** 对于容量、冲击成本、执行效率要求较高的策略，这部分产品需要谨慎对待。
6. **按资产类别看，{top_asset_type} ETF 成交额占比最高，达到 {top_asset_turnover_share:.2%}。** 当前阶段资金更集中在这类产品上，说明它们更能承接大额交易。
7. **全市场按近 20 日均成交额排名第一的 ETF 是 {top_20d_name}，约 {top_20d_turnover:.2f} 亿元。** 用近 20 日均值而不是单日值，更能反映稳定流动性。
8. **分 asset_type 看，各类内部的头部产品分别是：{asset_type_leaders_text}。** 这有助于在同一大类资产中快速找到核心交易载体。
9. **对股票 ETF 再细分后，各子类别的头部产品分别是：{stock_subtype_leaders_text}。** 这一步可以进一步区分宽基、行业、风格和主题内部的流动性核心。

### 如何使用这份结果

- 如果目标是找**稳定承接大资金**的 ETF，优先看 `top_20d_avg`、`asset_type_top10` 和 `stock_subtype_top10`。
- 如果目标是研究**市场结构**，重点看 `asset_type_summary`、`stock_subtype_summary` 和 `manager_summary`。
- 如果后续只想调整分析口径，可以直接读取 `{raw_data_dir}` 下的 CSV，不需要再重复抓取。只有在需要更新到新交易日时，才把 `REFRESH_SOURCE_DATA` 改成 `True`。
"""

display(Markdown(analysis_md))



### 核心结论

1. **本次分析基于 20260514 的快照，源数据来自 本地 CSV 缓存。**
2. **场内 ETF 共 1518 只，其中 1513 只有成交，覆盖率 99.67%。** 这说明 ETF 市场整体上是“广覆盖”的，大多数产品在最新交易日都有成交记录。
3. **最新交易日 ETF 总成交额约 4868.31 亿元，但单只 ETF 成交额中位数只有 0.23 亿元。** 总量很大，但典型单只产品的流动性并不高，市场呈现明显长尾。
4. **前 10 大 ETF 占全市场成交额 22.58%，前 50 大占 60.05%。** 这意味着流动性高度集中在头部产品，研究和交易时不能只看“是否上市”，还要看是否进入核心流动性池。
5. **有 73.63% 的有成交 ETF 当日成交额低于 1 亿元。** 对于容量、冲击成本、执行效率要求较高的策略，这部分产品需要谨慎对待。
6. **按资产类别看，债券 ETF 成交额占比最高，达到 42.06%。** 当前阶段资金更集中在这类产品上，说明它们更能承接大额交易。
7. **全市场按近 20 日均成交额排名第一的 ETF 是 短融ETF海富通，约 463.86 亿元。** 用近 20 日均值而不是单日值，更能反映稳定流动性。
8. **分 asset_type 看，各类内部的头部产品分别是：债券由 短融ETF海富通 领跑（近20日均成交额 463.86 亿元）；商品由 黄金ETF华安 领跑（近20日均成交额 37.54 亿元）；股票由 A500ETF华泰柏瑞 领跑（近20日均成交额 59.13 亿元）；货币由 银华日利ETF 领跑（近20日均成交额 174.37 亿元）；跨境由 中韩半导体ETF华泰柏瑞 领跑（近20日均成交额 83.93 亿元）。** 这有助于在同一大类资产中快速找到核心交易载体。
9. **对股票 ETF 再细分后，各子类别的头部产品分别是：主题ETF由 稀有金属ETF嘉实 领跑（近20日均成交额 4.80 亿元）；宽基指数ETF由 A500ETF华泰柏瑞 领跑（近20日均成交额 59.13 亿元）；行业ETF由 香港证券ETF易方达 领跑（近20日均成交额 53.10 亿元）；风格策略ETF由 红利低波ETF华泰柏瑞 领跑（近20日均成交额 6.74 亿元）。** 这一步可以进一步区分宽基、行业、风格和主题内部的流动性核心。

### 如何使用这份结果

- 如果目标是找**稳定承接大资金**的 ETF，优先看 `top_20d_avg`、`asset_type_top10` 和 `stock_subtype_top10`。
- 如果目标是研究**市场结构**，重点看 `asset_type_summary`、`stock_subtype_summary` 和 `manager_summary`。
- 如果后续只想调整分析口径，可以直接读取 `C:\Users\chern\Desktop\my_repos\vibetf\data\raw` 下的 CSV，不需要再重复抓取。只有在需要更新到新交易日时，才把 `REFRESH_SOURCE_DATA` 改成 `True`。


## 10. 保存分析结果

最后一步把分析结果导出成 CSV，方便后续在别的 notebook、脚本或 BI 工具里继续使用。

这里导出的文件属于**分析结果层**，和前面的**源数据层**分开：
- 源数据层在 `data/raw`
- 分析结果层在 `research/outputs`


In [11]:
snapshot.sort_values('avg_amount_100m_20d', ascending=False).to_csv(output_dir / f'etf_liquidity_snapshot_{latest_trade_date}.csv', index=False, encoding='utf-8-sig')
asset_type_summary.to_csv(output_dir / f'etf_liquidity_by_asset_type_{latest_trade_date}.csv', encoding='utf-8-sig')
asset_type_top10.to_csv(output_dir / f'etf_top10_by_asset_type_{latest_trade_date}.csv', index=False, encoding='utf-8-sig')
stock_subtype_summary.to_csv(output_dir / f'stock_etf_liquidity_by_subtype_{latest_trade_date}.csv', encoding='utf-8-sig')
stock_subtype_top10.to_csv(output_dir / f'stock_etf_top10_by_subtype_{latest_trade_date}.csv', index=False, encoding='utf-8-sig')
manager_summary.to_csv(output_dir / f'etf_liquidity_by_manager_{latest_trade_date}.csv', encoding='utf-8-sig')
top_20d_avg.to_csv(output_dir / f'etf_top10_by_20d_avg_{latest_trade_date}.csv', index=False, encoding='utf-8-sig')

for obsolete_name in [
    f'etf_top_current_liquidity_{latest_trade_date}.csv',
    f'etf_top_20d_avg_liquidity_{latest_trade_date}.csv',
]:
    obsolete_path = output_dir / obsolete_name
    if obsolete_path.exists():
        obsolete_path.unlink()

print(f'研究结果已输出到: {output_dir}')


研究结果已输出到: C:\Users\chern\Desktop\my_repos\vibetf\research\outputs
